In [ ]:
import mlflow
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [3]:
db_path = os.path.abspath("mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
mlflow.set_experiment("model_testing")

Tracking URI: sqlite:////Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/notebooks/mlflow.db


2026/08/24 09:59:45 INFO mlflow.tracking.fluent: Experiment with name 'model_testing' does not exist. Creating a new experiment.


<Experiment: artifact_location=('/Users/abhaybapat/Desktop/Data Science '
 'Project/betting_classification_model/notebooks/mlruns/21'), creation_time=1787583585790, experiment_id='21', last_update_time=1787583585790, lifecycle_stage='active', name='model_testing', tags={}, trace_location=None, workspace='default'>

In [4]:
df = pd.read_csv("../data/final_dataset.csv", index_col=0)

In [5]:
df["game_date"] = pd.DataFrame(df["game_date"])

df = df.sort_values(by="game_date")

In [6]:
df.columns

Index(['game_date', 'teamName_home', 'teamName_away', 'pre_game_elo_home',
       'is_B2B_home', 'pre_game_elo_away', 'is_B2B_away', 'pre_game_elo_diff',
       'days_rest_diff', 'possessions_rolling_diff', 'eFG_rolling_diff',
       'TO%_rolling_diff', 'OREB%_rolling_diff', 'FTR_rolling_diff',
       'off_rating_rolling_diff', 'def_rating_rolling_diff',
       'net_rating_rolling_diff', 'win_home'],
      dtype='object')

In [7]:
X = df.drop(columns=['game_date', 'teamName_home', 'teamName_away', 'win_home'])
y = df["win_home"]

In [9]:
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx].astype(int), y.iloc[split_idx:].astype(int)

models = {
    "LR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=.01, random_state=42))
    ]),
    "RF": RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ),
    "XGB": XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    "LGBM": LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),
}

results = []

for model_name, model in models.items():
    with mlflow.start_run(run_name=f"{model_name}_baseline"):
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        ll = log_loss(y_test, y_prob)

        mlflow.log_param("model_name", model_name)
        mlflow.log_param("split", "time_80_20")
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("log_loss", ll)

        results.append({
            "model": model_name,
            "accuracy": acc,
            "log_loss": ll,
        })

results_df = pd.DataFrame(results).sort_values(by="log_loss", ascending=True)
results_df

,model,accuracy,log_loss
0,LR,0.667364,0.609647
1,RF,0.663479,0.619381
2,XGB,0.652720,0.622076
3,LGBM,0.649432,0.629697


mlflow ui --backend-store-uri "sqlite:////Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/notebooks/mlflow.db" --port 5001